# 🤖 RAG SAF — AmorSaúde v3

Este notebook tem **duas fases independentes**:

| Fase | Quando rodar | O que faz |
|------|-------------|----------|
| 🔨 **BUILD** (células 1–5) | Só quando a base mudar | Lê JSON → gera embeddings → salva FAISS no Drive |
| 💬 **CHAT** (células 6–9) | Todo dia | Carrega FAISS salvo → responde perguntas |

> **Regra de ouro:** após qualquer alteração no `saf_chunks_otimizados.json`, rode o BUILD inteiro.
> No dia a dia, pule direto para o CHAT.

---
## ⚙️ SETUP — rode sempre (1x por sessão)
Instala dependências e configura credenciais.

In [1]:
!pip install -q langchain langchain-community langchain-groq \
               langchain-core faiss-cpu sentence-transformers \
               huggingface-hub

print('✅ Dependências ok!')

✅ Dependências ok!


In [2]:
import os


try:
    from google.colab import userdata
    os.environ['GROQ_API_KEY'] = userdata.get('GROQ_API_KEY')
    print('✅ GROQ_API_KEY carregada dos Secrets do Colab!')
except Exception:
    os.environ['GROQ_API_KEY'] = 'cole_sua_chave_aqui'
    print('⚠️  Usando chave manual — NÃO commite esse notebook!')

# ── Caminhos ────────────────────────────────────────────────────
FAISS_PATH  = '/content/drive/MyDrive/rag-saude/faiss_saf_v3'
CHUNKS_FILE = 'saf_chunks_otimizados.json'  # ajuste se necessário

print(f'📁 FAISS path: {FAISS_PATH}')
print(f'📄 Chunks file: {CHUNKS_FILE}')

✅ GROQ_API_KEY carregada dos Secrets do Colab!
📁 FAISS path: /content/drive/MyDrive/rag-saude/faiss_saf_v3
📄 Chunks file: saf_chunks_otimizados.json


In [3]:
from langchain_community.embeddings import HuggingFaceEmbeddings

print('🔧 Carregando intfloat/multilingual-e5-large (~560MB)...')

embeddings = HuggingFaceEmbeddings(
    model_name='intfloat/multilingual-e5-large',
    model_kwargs={'device': 'cpu'},
    encode_kwargs={'normalize_embeddings': True}
)

# sanity check
dim = len(embeddings.embed_query('teste'))
print(f'✅ Embeddings prontos! Dimensão do vetor: {dim}')
assert dim == 1024, f'Esperado 1024, obtido {dim}'

🔧 Carregando intfloat/multilingual-e5-large (~560MB)...


/tmp/ipykernel_32424/439835824.py:5: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

XLMRobertaModel LOAD REPORT from: intfloat/multilingual-e5-large
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ Embeddings prontos! Dimensão do vetor: 1024


---
# 🔨 FASE BUILD
**Rode apenas quando o JSON mudar.**

Fluxo:
```
saf_chunks_otimizados.json
    │
    ├─ load_json_chunks()   →  61 Documents com metadados ricos
    │
    ├─ FAISS.from_documents()  →  índice vetorial
    │
    └─ save_local(FAISS_PATH)  →  persiste no Drive
```

In [4]:
# ── BUILD CÉLULA 1: loader do JSON ──────────────────────────────
import json
from langchain_core.documents import Document

def load_json_chunks(filepath: str) -> list:
    """
    Lê o JSON estruturado e retorna uma lista de LangChain Documents.

    page_content  →  texto que será embedado
                     (título + conteúdo + faq = máximo recall semântico)

    metadata      →  campos estruturados passados ao LLM como contexto
                     e usados futuramente para filtros
    """
    with open(filepath, 'r', encoding='utf-8') as f:
        chunks = json.load(f)

    # campos de instrução que podem existir no chunk
    CAMPOS_INSTRUCAO = [
        'caminho_sistema', 'solucao', 'fluxo_resolucao', 'como_verificar',
        'como_reativar', 'passos', 'etapas_sequenciais', 'regras',
        'condicoes', 'formas_encerramento', 'fluxo_por_periodo',
        'checklist_verificacao', 'solucao_cache', 'regra_ambiguidade',
        'causa_raiz', 'instrucoes_fora_horario', 'mapeamento_chamados',
        'mapeamento_acoes', 'como_localizar', 'quando_usar',
    ]

    documents = []
    for chunk in chunks:

        # ── page_content: máximo sinal semântico para embedding ──────
        partes = []

        if chunk.get('titulo'):
            partes.append(f"TÓPICO: {chunk['titulo']}")

        if chunk.get('conteudo'):
            partes.append(chunk['conteudo'])

        for campo in CAMPOS_INSTRUCAO:
            val = chunk.get(campo)
            if not val:
                continue
            if isinstance(val, list):
                # lista de strings ou dicts
                texto = []
                for item in val:
                    if isinstance(item, dict):
                        texto.append(' '.join(str(v) for v in item.values()))
                    else:
                        texto.append(str(item))
                partes.append(' '.join(texto))
            elif isinstance(val, dict):
                partes.append(' '.join(str(v) for v in val.values()))
            else:
                partes.append(str(val))

        if chunk.get('restricoes'):
            partes.append('RESTRIÇÕES: ' + ' | '.join(chunk['restricoes']))

        if chunk.get('faq'):
            partes.append('PERGUNTAS: ' + ' | '.join(chunk['faq']))

        page_content = '\n'.join(p for p in partes if p)

        # ── metadata: campos para contexto no LLM e filtragem futura ─
        metadata = {
            'id':            chunk.get('id', ''),
            'titulo':        chunk.get('titulo', ''),
            'sistema':       ', '.join(chunk.get('sistema', [])),
            'categoria':     chunk.get('categoria', ''),
            'subcategoria':  chunk.get('subcategoria', ''),
            'acoes_zeev':    ', '.join(chunk.get('acoes_zeev', [])),
            'restricoes':    ' | '.join(chunk.get('restricoes', [])),
            'tem_caso_real': len(chunk.get('casos_validados', [])) > 0,
        }

        documents.append(Document(page_content=page_content, metadata=metadata))

    return documents


# ── Upload do JSON ───────────────────────────────────────────────
from google.colab import files
print('📎 Upload o arquivo: saf_chunks_otimizados.json')
uploaded = files.upload()
CHUNKS_FILE = [f for f in uploaded if f.endswith('.json')][0]

docs = load_json_chunks(CHUNKS_FILE)

print(f'\n✅ {len(docs)} chunks carregados')
print(f'   Exemplo — título:    {docs[3].metadata["titulo"]}')
print(f'   Exemplo — sistema:   {docs[3].metadata["sistema"]}')
print(f'   Exemplo — categoria: {docs[3].metadata["categoria"]}')
print(f'   Exemplo — acoes_zeev:{docs[3].metadata["acoes_zeev"]}')
print(f'\n   page_content[:300]:')
print(f'   {docs[3].page_content[:300]}')

📎 Upload o arquivo: saf_chunks_otimizados.json


Saving saf_chunks_otimizados.json to saf_chunks_otimizados (1).json

✅ 61 chunks carregados
   Exemplo — título:    Como alterar porcentagem ou valor de comissão de profissional
   Exemplo — sistema:   amei, webdental
   Exemplo — categoria: financeiro
   Exemplo — acoes_zeev:A071

   page_content[:300]:
   TÓPICO: Como alterar porcentagem ou valor de comissão de profissional
Para alterar o percentual de comissão ou valores de repasse de profissionais já cadastrados, a clínica deve abrir um chamado no Zeev utilizando o fluxo A071 — Manutenção de custos de profissionais e fornecedores. Esse fluxo é váli


In [5]:
# ── BUILD CÉLULA 2: gerar FAISS e salvar no Drive ────────────────
from langchain_community.vectorstores import FAISS
from google.colab import drive

drive.mount('/content/drive')

print(f'🗂️  Gerando embeddings e indexando {len(docs)} chunks no FAISS...')
print('   (pode levar 3–5 min no Colab CPU)')

vectorstore = FAISS.from_documents(docs, embeddings)

print(f'✅ {vectorstore.index.ntotal} vetores criados!')

vectorstore.save_local(FAISS_PATH)
print(f'✅ Índice salvo em: {FAISS_PATH}')
print()
print('─' * 50)
print('BUILD concluído. Agora vá para a FASE CHAT.')
print('─' * 50)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
🗂️  Gerando embeddings e indexando 61 chunks no FAISS...
   (pode levar 3–5 min no Colab CPU)
✅ 61 vetores criados!
✅ Índice salvo em: /content/drive/MyDrive/rag-saude/faiss_saf_v3

──────────────────────────────────────────────────
BUILD concluído. Agora vá para a FASE CHAT.
──────────────────────────────────────────────────


In [6]:
# ── BUILD CÉLULA 3 (opcional): validar índice recém criado ───────
print('🔍 Validando índice...')

queries_teste = [
    ('Como alterar comissão do dentista?',  'webdental'),
    ('Como abrir agenda médica no Amei?',   'amei'),
    ('Como acessar o Octadesk?',            'octadesk'),
    ('Qual chamado Zeev para cadastrar médico?', 'zeev'),
]

print(f'\n{'QUERY':<45} {'TOP RESULTADO':<45} {'SISTEMA'}')
print('─' * 110)

for query, sistema_esperado in queries_teste:
    results = vectorstore.similarity_search(query, k=1)
    top = results[0]
    titulo   = top.metadata['titulo'][:43]
    sistema  = top.metadata['sistema']
    ok = '✅' if sistema_esperado in sistema else '⚠️ '
    print(f'{ok} {query:<43} {titulo:<45} {sistema}')

print('\nSe todos ✅, índice está consistente.')

🔍 Validando índice...

QUERY                                         TOP RESULTADO                                 SISTEMA
──────────────────────────────────────────────────────────────────────────────────────────────────────────────
✅ Como alterar comissão do dentista?          Como alterar porcentagem ou valor de comiss   amei, webdental
✅ Como abrir agenda médica no Amei?           Amei — Abertura de agenda médica (grade de    amei
✅ Como acessar o Octadesk?                    Como recuperar acesso ao Octadesk             octadesk
✅ Qual chamado Zeev para cadastrar médico?    Priorização de chamados no SAF — Triagem co   zeev, octadesk, yungas

Se todos ✅, índice está consistente.


---
# 💬 FASE CHAT
**Rode todo dia.** Parte do FAISS já salvo no Drive.

Fluxo:
```
FAISS_PATH (Drive)
    │
    ├─ load_local()   →  vectorstore em memória
    │
    ├─ retriever MMR  →  top-5 chunks mais relevantes e diversos
    │
    ├─ format_docs()  →  contexto rico com metadados para o LLM
    │
    ├─ PROMPT_SAF     →  regras de comportamento do assistente
    │
    └─ Groq LLaMA-3.3-70b  →  resposta final
```

In [7]:
# ── CHAT CÉLULA 1: carregar índice ──────────────────────────────
from langchain_community.vectorstores import FAISS
from google.colab import drive

drive.mount('/content/drive')

vectorstore = FAISS.load_local(
    FAISS_PATH,
    embeddings,
    allow_dangerous_deserialization=True
)

print(f'✅ Índice carregado: {vectorstore.index.ntotal} vetores')
print(f'   Modelo: intfloat/multilingual-e5-large (dim=1024)')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Índice carregado: 61 vetores
   Modelo: intfloat/multilingual-e5-large (dim=1024)


In [8]:
# ── CHAT CÉLULA 2: montar pipeline ──────────────────────────────
from langchain_groq import ChatGroq
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

# ── Retriever MMR ────────────────────────────────────────────────
# MMR evita retornar chunks quase idênticos
# fetch_k=20: avalia 20 candidatos, retorna os 5 mais diversos e relevantes
retriever = vectorstore.as_retriever(
    search_type='mmr',
    search_kwargs={'k': 5, 'fetch_k': 20, 'lambda_mult': 0.7}
)

# ── LLM ─────────────────────────────────────────────────────────
llm = ChatGroq(
    model='llama-3.3-70b-versatile',
    api_key=os.environ['GROQ_API_KEY'],
    temperature=0.1
)

# ── Prompt SAF ───────────────────────────────────────────────────
PROMPT_SAF = PromptTemplate(
    input_variables=['context', 'question'],
    template="""Você é o assistente SAF da AmorSaúde, suporte operacional para franquias.

REGRAS — siga sempre:
1. Responda APENAS com base no CONTEXTO abaixo. Nunca invente.
2. Nunca misture sistemas: Medicina = Amei | Odontologia = WebDental.
3. Nunca invente códigos Zeev, e-mails ou processos fora do contexto.
4. Nunca justifique erros como "atraso entre sistema e aplicativo".
5. Se não encontrar a informação: "Não localizei essa informação na base.
   Entre em contato com o SAF: (16) 2018-2330 | saf@amorsaude.com"

FORMATO:
- Abra direto com a solução: "Para [objetivo]..."
- Seja técnico e objetivo. Cite caminhos de menu e códigos Zeev quando disponíveis.
- Se houver restrições, cite ao final em tópico "⚠️ Atenção:".
- Encerre com: "Posso ajudar com mais alguma coisa?"

CONTEXTO:
{context}

PERGUNTA: {question}

RESPOSTA:"""
)

# ── format_docs: contexto rico para o LLM ────────────────────────
def format_docs(retrieved_docs: list) -> str:
    partes = []
    for i, doc in enumerate(retrieved_docs, 1):
        m = doc.metadata

        linhas = [
            f"[Fonte {i}]",
            f"Título:    {m.get('titulo', '')}",
            f"Sistema:   {m.get('sistema', '')}",
            f"Categoria: {m.get('categoria', '')}",
        ]

        if m.get('acoes_zeev'):
            linhas.append(f"Zeev:      {m['acoes_zeev']}")

        if m.get('restricoes'):
            linhas.append(f"Restrições: {m['restricoes']}")

        linhas.append('')  # linha em branco
        linhas.append(doc.page_content)

        partes.append('\n'.join(linhas))

    return ('\n' + '━' * 60 + '\n').join(partes)


# ── Pipeline final ───────────────────────────────────────────────
qa_chain = (
    {'context': retriever | format_docs, 'question': RunnablePassthrough()}
    | PROMPT_SAF
    | llm
    | StrOutputParser()
)

print('🚀 Pipeline pronto!')

🚀 Pipeline pronto!


In [9]:
# ── CHAT CÉLULA 3: funções de consulta ──────────────────────────

def ask(question: str):
    """Resposta limpa — uso no dia a dia."""
    print(f'❓ {question}')
    print('─' * 60)
    print(qa_chain.invoke(question))
    print()


def ask_debug(question: str):
    """
    Resposta + quais chunks foram usados.
    Use quando a resposta parecer errada — revela o que o retriever trouxe.
    """
    print(f'❓ {question}')
    print('─' * 60)
    print(qa_chain.invoke(question))

    print('\n📎 Chunks recuperados:')
    for i, doc in enumerate(retriever.invoke(question), 1):
        m = doc.metadata
        tem_caso = '🧪' if m.get('tem_caso_real') else '  '
        zeev = f" | Zeev: {m['acoes_zeev']}" if m.get('acoes_zeev') else ''
        print(f'  {tem_caso} [{i}] {m["titulo"]}')
        print(f'        Sistema: {m["sistema"]} | Cat: {m["categoria"]}{zeev}')
    print()


def ask_score(question: str, k: int = 5):
    """
    Scores de similaridade L2 (menor = mais similar no FAISS).
    Útil para calibrar se o retriever está achando os chunks certos.
    """
    print(f'🔍 Scores para: "{question}"\n')
    results = vectorstore.similarity_search_with_score(question, k=k)
    for doc, score in results:
        m = doc.metadata
        barra = '█' * max(1, int((1 / (1 + score)) * 30))
        print(f'  {score:.4f} {barra}')
        print(f'  {m["titulo"]}')
        print(f'  {m["sistema"]} | {m["categoria"]}\n')


print('✅ Funções prontas!')
print('   ask(q)         → resposta limpa')
print('   ask_debug(q)   → resposta + chunks usados')
print('   ask_score(q)   → scores de similaridade')

✅ Funções prontas!
   ask(q)         → resposta limpa
   ask_debug(q)   → resposta + chunks usados
   ask_score(q)   → scores de similaridade


---
## 🧪 Testes rápidos
Rode após cada BUILD para confirmar que o índice está respondendo corretamente.

In [10]:
# Testes de sanidade — cobre Medicina, Odontologia e sistemas de chamado
testes = [
    'Como alterar a comissão do dentista no WebDental?',
    'Como abrir agenda de médico no Amei?',
    'Perdi o acesso ao Octadesk, o que fazer?',
    'Qual chamado Zeev para cadastrar novo profissional médico?',
    'O procedimento não aparece na grade após chamado no Zeev',
]

for q in testes:
    ask(q)
    print('=' * 60 + '\n')

❓ Como alterar a comissão do dentista no WebDental?
────────────────────────────────────────────────────────────
Para alterar a comissão do dentista no WebDental, é necessário abrir um chamado no Zeev utilizando o fluxo A071 — Manutenção de custos de profissionais e fornecedores. Esse fluxo é válido tanto para o sistema Amei quanto para o WebDental.

⚠️ Atenção: O SAF não realiza essa alteração manualmente — é obrigatório o uso do Zeev A071.

Posso ajudar com mais alguma coisa?


❓ Como abrir agenda de médico no Amei?
────────────────────────────────────────────────────────────
Para abrir a agenda de um médico no Amei, acessar: Cadastro > Lista de Profissionais > selecionar o profissional > Horários de Atendimento > clicar em Incluir. Isso permitirá configurar os horários de atendimento do profissional.

⚠️ Atenção: Certifique-se de estar no sistema Amei e não no WebDental, pois a agenda é um termo exclusivo da Medicina.

Posso ajudar com mais alguma coisa?


❓ Perdi o acesso ao Octade

In [11]:
# Debug: inspecione chunks recuperados quando uma resposta parecer estranha
ask_debug('Como abrir cadeira para dentista?')

❓ Como abrir cadeira para dentista?
────────────────────────────────────────────────────────────
Para abrir cadeira para dentista no WebDental, acesse o menu "Configurações" e, em seguida, selecione "Agenda" e, dentro dessa seção, escolha "Cadeiras". Nessa área, você poderá configurar e abrir a cadeira para o dentista, definindo os horários de atendimento e outras configurações necessárias.

⚠️ Atenção: Certifique-se de que está utilizando o sistema WebDental, pois a gestão de cadeiras é exclusiva da Odontologia e não se aplica ao sistema Amei, que é utilizado para a Medicina.

Posso ajudar com mais alguma coisa?

📎 Chunks recuperados:
     [1] WebDental — Abertura de cadeira/agenda odontológica
        Sistema: webdental | Cat: agenda_odontologica
     [2] Distinção crítica: Agenda (Medicina/Amei) vs Cadeira (Odontologia/WebDental)
        Sistema: amei, webdental | Cat: regras_criticas
     [3] Amei — Módulo Financeiro: caixas e cadeado
        Sistema: amei | Cat: financeiro
     [4

In [ ]:
# Scores: útil para comparar antes/depois de mudar o JSON ou embeddings
ask_score('inativação de profissional médico')

---
## 🔁 Loop interativo
Conversa em tempo real com o assistente SAF. Digite `sair` para encerrar.

In [12]:
print('🤖 Assistente SAF — AmorSaúde')
print('💡 Dica: perguntas em português natural funcionam melhor')
print('   Digite "sair" para encerrar.\n')

while True:
    try:
        pergunta = input('Você: ').strip()
    except (EOFError, KeyboardInterrupt):
        print('\n👋 Até logo!')
        break

    if not pergunta:
        continue
    if pergunta.lower() in ('sair', 'exit', 'quit', 'q'):
        print('👋 Até logo!')
        break
    if pergunta.lower() == 'debug':
        pergunta = input('Pergunta para debug: ').strip()
        ask_debug(pergunta)
    else:
        ask(pergunta)

🤖 Assistente SAF — AmorSaúde
💡 Dica: perguntas em português natural funcionam melhor
   Digite "sair" para encerrar.

Você: Não consigo abrir agenda de uma médica. O quefazer?
❓ Não consigo abrir agenda de uma médica. O quefazer?
────────────────────────────────────────────────────────────
Para abrir a agenda de uma médica no sistema Amei, acesse: Cadastro > Lista de Profissionais > selecione a médica > Horários de Atendimento > clique em Incluir. Certifique-se de que a grade de horários esteja corretamente configurada e que os dias da semana em que a médica atende estejam marcados (flegados).

⚠️ Atenção: Verifique se a médica está ativa no sistema e se não há bloqueios de agenda que possam impedir a abertura da agenda.

Posso ajudar com mais alguma coisa?

Você: sair
👋 Até logo!


---
## 📋 Referência rápida

### Quando rodar cada fase

| Situação | O que fazer |
|----------|-------------|
| Primeira vez | SETUP + BUILD completo + CHAT |
| Nova sessão Colab (JSON não mudou) | SETUP + CHAT (célula load) |
| JSON atualizado | SETUP + BUILD completo + CHAT |
| Só testar uma pergunta | SETUP + CHAT a partir da célula load |

### Diagnóstico de problemas

| Sintoma | Causa provável | Solução |
|---------|---------------|---------|
| Resposta genérica demais | Chunks errados no retriever | `ask_debug()` para inspecionar |
| Sistema errado na resposta (Amei/WebDental trocados) | Chunk sem metadado `sistema` | Verificar JSON + rebuild |
| Score alto (> 1.0) em tudo | FAISS antigo com modelo diferente | Rebuild obrigatório |
| `InvalidIndexException` no load | FAISS salvo com dimensão diferente | Rebuild com e5-large |

### Modelo de embeddings
```
intfloat/multilingual-e5-large
├── Dimensão: 1024
├── Línguas:  100+ incluindo PT-BR
└── Tamanho:  ~560MB
```

### Parâmetros do retriever MMR
```python
k=5          # chunks retornados ao LLM
fetch_k=20   # candidatos avaliados pelo MMR antes de filtrar
lambda_mult=0.7  # 1.0 = só relevância | 0.0 = só diversidade
```

In [1]:
import os

print("Caminho atual:", os.getcwd())
print("FAISS existe:", os.path.exists("faiss_index"))
print("DATA existe:", os.path.exists("data"))

Caminho atual: C:\Users\giovanna.ribeiro\OneDrive - AmorSaúde\Área de Trabalho\RAG SAF\notebook
FAISS existe: False
DATA existe: False


In [2]:
import os

FAISS_PATH = '../faiss_index'
CHUNKS_FILE = '../data/saf_chunks_otimizados.json'

print("FAISS existe:", os.path.exists(FAISS_PATH))
print("DATA existe:", os.path.exists(CHUNKS_FILE))

FAISS existe: True
DATA existe: True
